## Concept 11 — Multi-hop RAG

### What is it?
Some questions cannot be answered in a single retrieval. They require **chaining multiple lookups** where each answer informs the next question.

### Pipeline
```
Complex Query
  → Decompose into sub-questions: Q1, Q2, Q3
  ↓
  Hop 1: Retrieve + Answer Q1
  ↓
  Hop 2: Use Q1 answer to form Q2 → Retrieve + Answer Q2
  ↓
  Hop 3: Use Q2 answer to form Q3 → Retrieve + Answer Q3
  ↓
  Compose final answer from all hops
```

### Example
```
Complex: 'What storage backs the mechanism LangSmith uses to resume interrupted runs?'

Hop 1: 'What mechanism does LangSmith use to resume interrupted runs?'
       → Answer: 'Checkpoints'

Hop 2: 'What storage system backs checkpoints in LangSmith?'
       → Answer: 'PostgreSQL'

Final: 'PostgreSQL backs the checkpoint mechanism used to resume interrupted runs.'
```

### This is the most advanced RAG pattern
Handles questions that no other RAG technique can answer in one pass.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, run_queries

chunks, vectorstore, embeddings, llm, prompt = setup()
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Multi-hop needs complex questions that require chaining
MULTIHOP_QUERIES = [
    "What storage system backs the mechanism LangSmith uses to resume interrupted runs?",
    "What types of data does the default persistence layer in LangSmith store and what database does it use?"
]

### Step 1 — Decompose Complex Query into Sub-questions
LLM breaks a complex question into a chain of simpler sub-questions.

In [ ]:
def decompose_query(query, n_hops=3):
    decompose_prompt = (
        f"Break this complex question into {n_hops} simpler sub-questions that can be answered step by step.\n"
        f"Each sub-question should build on the answer of the previous one.\n"
        f"Return ONLY the sub-questions, one per line, no numbering, no extra text.\n\n"
        f"Complex Question: {query}"
    )
    response = llm.invoke(decompose_prompt).content.strip()
    sub_qs   = [q.strip() for q in response.split('\n') if q.strip()]
    return sub_qs[:n_hops]

# Test decomposition
query  = MULTIHOP_QUERIES[0]
sub_qs = decompose_query(query)
print(f"Complex query: {query}\n")
print("Decomposed into:")
for i, q in enumerate(sub_qs, 1):
    print(f"  Hop {i}: {q}")

### Step 2 — Answer Each Hop
Retrieve and answer each sub-question, carrying forward previous findings as context.

In [ ]:
def answer_hop(question, previous_context=""):
    """Answer a sub-question, optionally with context from previous hops."""
    docs = retriever.invoke(question)
    retrieved_context = format_docs(docs)

    # Combine previous hop findings with newly retrieved docs
    if previous_context:
        combined_context = f"Previous findings:\n{previous_context}\n\nNew retrieved context:\n{retrieved_context}"
    else:
        combined_context = retrieved_context

    response = llm.invoke(prompt.format_messages(context=combined_context, question=question))
    return response.content

### Step 3 — Multi-hop Chain
Chain all hops together, each building on the previous.

In [ ]:
def multi_hop_rag(query, n_hops=3):
    print(f"Complex Query: {query}\n")

    # Decompose
    sub_questions = decompose_query(query, n_hops)
    print("Sub-questions:")
    for i, q in enumerate(sub_questions, 1):
        print(f"  Hop {i}: {q}")

    # Answer each hop, carrying forward findings
    cumulative_findings = ""
    for i, sub_q in enumerate(sub_questions, 1):
        print(f"\n--- Hop {i} ---")
        print(f"Q: {sub_q}")
        hop_answer = answer_hop(sub_q, cumulative_findings)
        print(f"A: {hop_answer}")
        # Append this hop's finding to carry forward
        cumulative_findings += f"\nQ: {sub_q}\nA: {hop_answer}"

    # Final answer using all hop findings
    print("\n--- Composing Final Answer ---")
    final_prompt = (
        f"Using these research findings from multiple steps:\n{cumulative_findings}\n\n"
        f"Now answer the original question: {query}"
    )
    final_answer = llm.invoke(final_prompt).content
    return final_answer

### Test — Multi-hop Queries
These questions require chaining multiple lookups — try the standard queries too to compare.

In [ ]:
for q in MULTIHOP_QUERIES:
    print("\n" + "="*60)
    answer = multi_hop_rag(q)
    print(f"\nFinal Answer: {answer}")
    print("="*60)

### Compare — Same Standard Queries Across All Notebooks
Run standard queries to see how Multi-hop handles questions that single-hop RAG answers fine too.

In [ ]:
from RAGutils import TEST_QUERIES
run_queries(multi_hop_rag, queries=TEST_QUERIES, label="Concept 11 — Multi-hop RAG")

### Congratulations!
You have now completed all 11 RAG concepts:

| # | Concept | Key Idea |
|---|---------|----------|
| 1 | Simple RAG | Manual pipeline — embed, search, answer |
| 2 | LCEL + MMR | Clean pipeline + diverse results |
| 3 | Reranking | Score candidates, keep best |
| 4 | Hybrid RAG | Vector + BM25 combined |
| 4B | Agentic RAG | Self-correction loop |
| 5 | RAG Fusion | Multiple queries + RRF merging |
| 6 | HyDE | Embed hypothetical answer for better search |
| 7 | Parent Doc | Small chunks for search, large for LLM |
| 8 | Contextual Compression | Strip noise before LLM |
| 9 | CRAG | Evaluate docs, fall back to web |
| 10 | Self-RAG | LLM controls every decision |
| 11 | Multi-hop | Chain multiple retrievals |

**Next steps:** Combine multiple techniques (e.g., Hybrid + Reranking + CRAG) for production-grade RAG.